# Axis 2 — Eviction Policy & Expert Locality (Windows dGPU / PyTorch)

**Question.** Does expert routing have enough temporal locality to justify smarter eviction than FIFO / LRU?

**Hypotheses (from the team page).**
* **H1** — LFU beats LRU on homogeneous, sustained work by anchoring the core experts.
* **H2** — During multi-turn domain shifts, *un-decayed* LFU keeps stale experts resident and falls below LRU.

**Router-score-aware policy (RSA).** For every resident expert, keep a *decayed sum of the router probability mass* it has received, and evict the smallest:

$$ s_e(t) = \sum_i g_{e,i}\; 2^{-(t - t_i)/H} $$

where $g_{e,i}$ is the gate weight the router gave expert *e* in call *i* (summed over that call's tokens), and *H* is a half-life in cache accesses. The policies are the shared ones in `moe.common`, identical on every device.

## Method (same as the Mac)
1. Two request streams from `prompts.json`: **sustained** (programming requests only) and **domain shift** (programming → creative → technical → math blocks).
2. **Real runs** at a headline K for every policy, on the real disk-backed VRAM cache, timed. All runs must produce identical text *and* identical routing.
3. **K sweep by exact replay** of the recorded access stream, with the cache's own grouping and pinning rules; replay is asserted to reproduce every real run exactly.
4. **Locality analysis** — frequency skew, reuse times, and how much the expert sets of different domains overlap.

Hit rates by replay depend only on routing, so they should nearly match the Mac's; real-run timings are this machine's (dGPU, NVMe, BF16). `MOE_SMOKE=1` runs a tiny configuration.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Shared code: the `moe` package in src/. `pip install -e ".[windows]"` installs it; the path
# line below makes the notebook work without installing.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'moe').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moe import common, torch_cache as tc, results as rs, figures as fg   # tc is Windows (PyTorch)
for m in (common, tc, rs, fg):
    importlib.reload(m)                # pick up edits to src/moe without restarting the kernel

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-2-eviction-policies'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-2-eviction-policies') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = 'windows-dgpu'                 # results file: <axis>/results/windows-dgpu.json
AXIS = 'axis-2-eviction-policies'
TABLES = {}                             # every table this run produces, saved at the end

GPU = tc.default_device()               # the dedicated GPU; K counts experts in its VRAM
assert GPU.type == 'cuda' or SMOKE, 'No CUDA device found: this notebook measures the dedicated GPU.'
torch.set_grad_enabled(False)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(tc.system_info(GPU))

## Configuration

In [ ]:
POLICIES = {                     # name -> (kind, kwargs); half-lives are in cache accesses
    'FIFO':      ('FIFO', {}),
    'LRU':       ('LRU', {}),
    'LFU':       ('LFU', {}),
    'LFU-H1024': ('LFU', {'half_life': 1024}),
    'LFU-H4096': ('LFU', {'half_life': 4096}),
    'RSA-H1024': ('RSA', {'half_life': 1024}),
    'RSA-H4096': ('RSA', {'half_life': 4096}),
}
REAL_K = 256                     # headline K: every policy runs on the real cache here
SWEEP_K = [128, 192, 256, 384, 512]
SUSTAINED_N = 12                 # programming requests, back to back
SHIFT_ORDER, SHIFT_N = ['programming', 'creative', 'technical', 'math'], 4
MAX_NEW_TOKENS = 48
POST_SHIFT_WINDOW = 2048         # first decode accesses after a switch (~16 tokens x 128)
ROLLING = 1024

if SMOKE:
    POLICIES = {k: POLICIES[k] for k in ['LRU', 'LFU', 'RSA-H1024']}
    SWEEP_K, SUSTAINED_N, SHIFT_N, MAX_NEW_TOKENS = [128, 256], 2, 1, 6
    POST_SHIFT_WINDOW, ROLLING = 256, 128

def policy(name, K):
    kind, kw = POLICIES[name]
    return tc.make_policy(kind, K, **kw)

print('policies:', list(POLICIES), '| real K:', REAL_K, '| sweep K:', SWEEP_K)
CONFIG = rs.config_from(globals())   # recorded in the results file

## Model (non-expert weights only) and correctness gate

In [ ]:
tc.download_model()        # allenai/OLMoE-1B-7B-0125-Instruct (BF16) into models/, only if it is missing
model, tokenizer, index = tc.load_model(device=GPU)
print(index.describe())
print(f'Non-expert weights resident on {GPU}: {tc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| expert reads: {tc.ExpertReader(index).mode}')
# Measured ONCE, before any expert bank exists: free VRAM read later in the run is unreliable.
SAFE_K = tc.max_safe_capacity(index, GPU)
print(f'Largest K that fits in the VRAM free right now: {SAFE_K} (bank {SAFE_K * index.expert_bytes / 1e9:.1f} GB)')

# Correctness gate: one layer through the K-slot bank vs the stock transformers layer
# (all 64 experts of that layer resident, eager expert loop). Must be bit-identical.
diff = tc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock transformers: max |diff| = {diff}')

## Request streams

In [ ]:
# Sustained: one homogeneous task type, request after request.
prompt_file = rs.load_prompts(AXIS_DIR)   # axis-2-eviction-policies/prompts.json, shared by every device
SUSTAINED = prompt_file['SUSTAINED']
# Domain shift: blocks of different kinds of request.
SHIFT = prompt_file['SHIFT']
SHIFT_DOMAINS = [d for d in SHIFT_ORDER if d in SHIFT]
STREAMS = {
    'sustained': tc.requests({'programming': SUSTAINED}, n_per_domain=SUSTAINED_N),
    'domain_shift': tc.requests({d: SHIFT[d] for d in SHIFT_DOMAINS}, n_per_domain=SHIFT_N, order='blocks'),
}
for name, reqs in STREAMS.items():
    print(f'{name:13s}: {len(reqs)} requests | blocks: {[r["domain"] for r in reqs][::max(1, SHIFT_N)]}')

## Real runs at K = REAL_K, one per policy
Each run starts from an empty VRAM cache and serves the whole stream. Timing is real; the hit/miss counts are what the replay must reproduce.

In [ ]:
assert REAL_K <= SAFE_K, f'REAL_K = {REAL_K} does not fit in free VRAM (largest safe K = {SAFE_K}).'
real, traces = [], {}
for sname, reqs in STREAMS.items():
    ref_text = ref_stream = None
    for pname in POLICIES:
        tc.free_memory()
        cache = tc.ExpertCache(index, REAL_K, policy(pname, REAL_K), device=GPU)
        print(f'\n[{sname}] {pname} @ K={REAL_K}')
        t0 = time.perf_counter()
        df, rec = tc.run_workload(model, tokenizer, cache, reqs, MAX_NEW_TOKENS, log=None)
        s = cache.summary()
        stream = [(e['layer'], tuple(e['eids'])) for e in rec.events]
        if ref_text is None:
            ref_text, ref_stream, traces[sname] = list(df.text), stream, rec.events
        assert list(df.text) == ref_text, 'policy changed the output!'
        assert stream == ref_stream, 'policy changed the routing!'
        sim = policy(pname, REAL_K)
        hits = tc.replay(rec.events, sim)
        assert int(hits.sum()) == s['hits'] and sim.evictions == s['evictions'], 'replay mismatch'
        dec = df.new_tokens.sum() - len(df)
        real.append({'stream': sname, 'policy': pname, 'K': REAL_K, 'hit_rate': s['hit_rate'],
                     'misses': s['misses'], 'GB_loaded': s['GB_loaded'], 'io_s': s['io_ms'] / 1000,
                     'disk_s': s['disk_s'], 'h2d_s': s['h2d_s'],
                     'wall_s': time.perf_counter() - t0, 'decode_tok_s': dec / df.decode_s.sum(),
                     'ttft_mean_s': df.ttft_s.mean()})
        print(f'  hit {s["hit_rate"]:.1%} | {s["misses"]} misses | io {s["io_ms"] / 1000:.1f}s | '
              f'decode {real[-1]["decode_tok_s"]:.2f} tok/s | replay exact, output identical')
        cache.close(); tc.detach_cache(model); del cache

real = pd.DataFrame(real)
TABLES['real_runs'] = real
for sname, ev in traces.items():
    tc.save_json(RESULTS / f'trace_{sname}.json', ev)      # git-ignored
print('\nAll policies: identical text, identical routing, replay == real cache.')

## K sweep by exact replay
Same stream, every policy, every K. Per-segment and post-shift hit rates use the domain tag of each access; post-shift counts only the first `POST_SHIFT_WINDOW` **decode** accesses of each new domain (prefill is dominated by compulsory misses).

In [ ]:
frames = {s: tc.events_frame(ev) for s, ev in traces.items()}
rows, rolling = [], {}
for sname, ev in traces.items():
    acc = frames[sname]
    dom = acc.prompt.map(lambda i: STREAMS[sname][i]['domain']).to_numpy()
    dec = (acc.phase == 'decode').to_numpy()
    for K in sorted(set(SWEEP_K) | {REAL_K}):
        for pname in POLICIES:
            hits = tc.replay(ev, policy(pname, K))
            row = {'stream': sname, 'K': K, 'policy': pname, 'hit_rate': hits.mean(),
                   'decode_hit_rate': hits[dec].mean()}
            if sname == 'domain_shift':
                post = []
                for d in SHIFT_DOMAINS:
                    row[f'seg_{d}'] = hits[dom == d].mean()
                    if d != SHIFT_DOMAINS[0]:
                        idx = np.flatnonzero((dom == d) & dec)[:POST_SHIFT_WINDOW]
                        post.append(hits[idx].mean())
                row['post_shift'] = float(np.mean(post))
            rows.append(row)
            if K == REAL_K:
                rolling[(sname, pname)] = np.convolve(hits.astype(float), np.ones(ROLLING) / ROLLING, 'valid')
sweep = pd.DataFrame(rows)
TABLES['sweep'] = sweep

for sname in traces:
    print(f'\n{sname}: hit rate by policy (rows) and K (columns)')
    display(sweep[sweep.stream == sname].pivot(index='policy', columns='K', values='hit_rate')
            .loc[list(POLICIES)].style.format('{:.1%}').highlight_max(axis=0, color='#c6efce'))

In [ ]:
d = sweep[(sweep.stream == 'domain_shift')]
print('Domain shift — first decode window after each switch (post-shift hit rate)')
display(d.pivot(index='policy', columns='K', values='post_shift').loc[list(POLICIES)]
        .style.format('{:.1%}').highlight_max(axis=0, color='#c6efce'))
print(f'Domain shift — per-segment hit rate at K={REAL_K}')
display(d[d.K == REAL_K].set_index('policy')[[f'seg_{x}' for x in SHIFT_DOMAINS]]
        .loc[list(POLICIES)].style.format('{:.1%}'))

## Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
for ax, sname in zip(axes, traces):
    d = sweep[sweep.stream == sname]
    for i, pname in enumerate(POLICIES):
        dd = d[d.policy == pname].sort_values('K')
        ax.plot(dd.K, dd.hit_rate, '-o', label=pname, lw=2.5 if pname.startswith('RSA') else 1.5)
    ax.set_xscale('log', base=2); ax.set_xticks(sorted(d.K.unique())); ax.set_xticklabels(sorted(d.K.unique()))
    ax.set_title(sname); ax.set_xlabel('global K')
axes[0].set_ylabel('hit rate'); axes[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

# Rolling hit rate along the domain-shift stream
acc = frames['domain_shift']
dom = acc.prompt.map(lambda i: STREAMS['domain_shift'][i]['domain']).to_numpy()
bounds = [int(np.flatnonzero(dom == d)[0]) for d in SHIFT_DOMAINS]
fig, ax = plt.subplots(figsize=(15, 4.5))
for pname in POLICIES:
    ax.plot(rolling[('domain_shift', pname)], label=pname, lw=1.8 if pname.startswith('RSA') else 1)
for b, d in zip(bounds, SHIFT_DOMAINS):
    ax.axvline(b, color='k', ls='--', alpha=0.4); ax.text(b, 0.97, f' {d}', va='top', fontsize=9)
ax.set_ylim(0, 1); ax.set_xlabel('cache access #'); ax.set_ylabel(f'rolling hit rate ({ROLLING})')
ax.set_title(f'Domain shift stream at K={REAL_K}'); ax.legend(ncol=4, fontsize=8)
plt.show()

# Real cost at the headline K
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, col, lab in [(axes[0], 'io_s', 'load time: disk + copy to VRAM (s)'), (axes[1], 'decode_tok_s', 'decode tok/s')]:
    p = real.pivot(index='policy', columns='stream', values=col).loc[list(POLICIES)]
    p.plot.bar(ax=ax, rot=30); ax.set_ylabel(lab); ax.set_title(f'Real runs at K={REAL_K}: {lab}')
fig.tight_layout(); plt.show()

## Locality analysis
A policy can only exploit structure that exists. Three views of the access stream:
* **Frequency skew** — how concentrated accesses are (Gini; share of the top 64 experts).
* **Reuse time** — accesses since the last request for the same expert. A K-slot cache can only hit re-uses that come back "soon enough".
* **Domain overlap** — Jaccard similarity of each domain's top-256 most-used experts (what a cache of that size would hold). Low overlap is what makes stale LFU counts costly after a shift.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
stats = []
for sname, acc in frames.items():
    keys = list(zip(acc.layer, acc.eid))
    cnt = np.array(sorted(pd.Series(keys).value_counts().values, reverse=True))
    stats.append({'stream': sname, 'accesses': len(keys), 'distinct': len(cnt), 'gini': tc.gini(cnt),
                  'top64_share': cnt[:64].sum() / cnt.sum(), 'top256_share': cnt[:256].sum() / cnt.sum()})
    axes[0].loglog(np.arange(1, len(cnt) + 1), cnt, label=sname)
    last, reuse = {}, []
    for i, k in enumerate(keys):
        if k in last:
            reuse.append(i - last[k])
        last[k] = i
    r = np.sort(reuse)
    axes[1].plot(r, np.arange(1, len(r) + 1) / len(keys), label=sname)
axes[0].set_xlabel('expert rank'); axes[0].set_ylabel('accesses'); axes[0].set_title('Frequency vs rank'); axes[0].legend()
axes[1].set_xscale('log'); axes[1].set_xlabel('reuse time (accesses)'); axes[1].set_ylabel('cumulative share of accesses')
axes[1].set_title('Reuse-time CDF (ceiling = 1 - compulsory)'); axes[1].legend()

acc = frames['domain_shift']
acc = acc.assign(domain=acc.prompt.map(lambda i: STREAMS['domain_shift'][i]['domain']))
# Nearly every expert is touched by every domain at least once, so compare what a
# cache would actually keep: each domain's TOP_N most-used experts.
TOP_N = 256
sets = {d: set(g.groupby(['layer', 'eid']).size().nlargest(TOP_N).index) for d, g in acc.groupby('domain')}
names = [d for d in SHIFT_DOMAINS if d in sets]
jac = np.array([[len(sets[a] & sets[b]) / len(sets[a] | sets[b]) for b in names] for a in names])
im = axes[2].imshow(jac, vmin=0, vmax=1, cmap='Blues')
axes[2].set_xticks(range(len(names))); axes[2].set_xticklabels(names, rotation=30)
axes[2].set_yticks(range(len(names))); axes[2].set_yticklabels(names)
for i in range(len(names)):
    for j in range(len(names)):
        axes[2].text(j, i, f'{jac[i, j]:.2f}', ha='center', va='center', fontsize=8)
axes[2].set_title(f'Overlap of each domain\'s top-{TOP_N} experts (Jaccard)')
fig.tight_layout(); plt.show()
pd.DataFrame(stats).style.format({'gini': '{:.2f}', 'top64_share': '{:.1%}', 'top256_share': '{:.1%}'})

## Hypothesis check (descriptive)

In [ ]:
def hr(stream, K, p, col='hit_rate'):
    return float(sweep[(sweep.stream == stream) & (sweep.K == K) & (sweep.policy == p)][col].iloc[0])

Ks = sorted(sweep.K.unique())
print('H1  sustained: LFU vs LRU')
for K in Ks:
    a, b = hr('sustained', K, 'LFU'), hr('sustained', K, 'LRU')
    print(f'   K={K:4d}: LFU {a:.1%} vs LRU {b:.1%}  ->  {"LFU better" if a > b else "LRU better or equal"} ({a - b:+.1%})')
print('\nH2  domain shift: un-decayed LFU vs LRU (overall | post-shift window)')
for K in Ks:
    a, b = hr('domain_shift', K, 'LFU'), hr('domain_shift', K, 'LRU')
    pa, pb = hr('domain_shift', K, 'LFU', 'post_shift'), hr('domain_shift', K, 'LRU', 'post_shift')
    print(f'   K={K:4d}: {a:.1%} vs {b:.1%} | post-shift {pa:.1%} vs {pb:.1%}  ->  '
          f'{"supports H2" if (a < b and pa < pb) else "does not support H2"}')
print('\nRouter-score-aware policy: rank among all policies (1 = best hit rate)')
for sname in traces:
    for K in Ks:
        d = sweep[(sweep.stream == sname) & (sweep.K == K)].set_index('policy').hit_rate.sort_values(ascending=False)
        rsa = [p for p in d.index if p.startswith('RSA')]
        best_rsa = max(rsa, key=lambda p: d[p])
        print(f'   {sname:13s} K={K:4d}: best RSA = {best_rsa} rank {list(d.index).index(best_rsa) + 1}/{len(d)} '
              f'({d[best_rsa]:.1%} vs best {d.index[0]} {d.iloc[0]:.1%})')
print('\nTreat differences of about a point as noise: one fixed prompt set, one seed.')

## Notes for reading the results

* **Within-device trends first.** These are PyTorch / BF16 / CUDA numbers. The Mac runs MLX / F16 on unified memory: compare the *shape* of the curves across devices, not absolute values as one controlled experiment.
* **No page cache to purge.** Expert reads are unbuffered (`reads` = `windows-unbuffered` in the system info), so every miss really goes to the disk. If the reader had to fall back to buffered reads it prints a warning and the mode says `buffered`.
* **K is the VRAM tier.** The timed K values are those whose bank fits in the VRAM free at load (`SAFE_K`); larger K are replay-only (hit rates exact, no timing). Close other GPU programs before a full run.

## Save results
Every table above goes into **one file for this device**: `results/windows-dgpu.json`, with the configuration, prompts and machine it ran on.
The Mac and Windows iGPU notebooks write `mac.json` / `windows-igpu.json` next to it in the same format (see `experiments/README.md`).

In [ ]:
CONFIG['SAFE_K'] = globals().get('SAFE_K')
path = rs.save(RESULTS, AXIS, DEVICE, TABLES, config=CONFIG, prompts=rs.load_prompts(AXIS_DIR),
               system=tc.system_info(GPU))
print('Saved', path.relative_to(ROOT), '| tables:', {k: len(v) for k, v in TABLES.items()})

## Cross-device figures
The report figures, drawn from every device's results file in this folder (colour = device). Only shown here;
`python -m moe.figures` writes them to `figures/` once all devices have results.

In [ ]:
figs = fg.make(AXIS, RESULTS)